In [2]:
import pandas as pd
from sklearn.metrics import cohen_kappa_score
from statsmodels.stats.inter_rater import fleiss_kappa, aggregate_raters

# ---- EDIT THESE PATHS ----
path_nadia   = "exit_sample_100_Nadia.xlsx"
path_llm     = "exit_sample_100_LLM.xlsx"
path_jihyun  = "exit_sample_100_JiHyun.xlsx"
# ---------------------------

label_col = "exit_intention_class"

def load(path, rater_name):
    df = pd.read_excel(path)
    return df[["post_id", label_col]].rename(columns={label_col: rater_name})

nadia  = load(path_nadia, "nadia")
llm    = load(path_llm, "llm")
jihyun = load(path_jihyun, "jihyun")

merged = nadia.merge(llm, on="post_id").merge(jihyun, on="post_id")
print(f"Merged rows (all three raters present): {len(merged)}")

# ---------- Pairwise Cohen's kappa ----------
kappa_llm_nadia   = cohen_kappa_score(merged["llm"], merged["nadia"])
kappa_llm_jihyun  = cohen_kappa_score(merged["llm"], merged["jihyun"])
kappa_jihyun_nadia = cohen_kappa_score(merged["jihyun"], merged["nadia"])

print("\n=== Cohen's Kappa (pairwise) ===")
print(f"LLM vs Nadia:      {kappa_llm_nadia:.4f}")
print(f"LLM vs Ji Hyun:    {kappa_llm_jihyun:.4f}")
print(f"Ji Hyun vs Nadia:  {kappa_jihyun_nadia:.4f}")

# ---------- Fleiss' kappa (three-way) ----------
ratings = merged[["nadia", "llm", "jihyun"]].values
agg, categories = aggregate_raters(ratings)
fk = fleiss_kappa(agg)

print("\n=== Fleiss' Kappa (Nadia, LLM, Ji Hyun) ===")
print(f"Fleiss' kappa: {fk:.4f}")

# Save summary
summary = pd.DataFrame({
    "metric": ["cohen_llm_nadia", "cohen_llm_jihyun", "cohen_jihyun_nadia", "fleiss_three_way"],
    "value": [kappa_llm_nadia, kappa_llm_jihyun, kappa_jihyun_nadia, fk],
})
summary.to_csv("kappa_summary.csv", index=False)
print("\nSaved to kappa_summary.csv")

Merged rows (all three raters present): 100

=== Cohen's Kappa (pairwise) ===
LLM vs Nadia:      0.8779
LLM vs Ji Hyun:    0.7856
Ji Hyun vs Nadia:  0.7660

=== Fleiss' Kappa (Nadia, LLM, Ji Hyun) ===
Fleiss' kappa: 0.8098

Saved to kappa_summary.csv
